# Clase 153 — MCP (Model Context Protocol)

Implementamos un **mini servidor y cliente MCP** en Python: JSON-RPC sobre stdin/stdout (simulado in-process). Sin dependencias externas.

In [ ]:
import json, re, uuid
import numpy as np
np.random.seed(42)

## 1. Por qué MCP > function calling proprietario

- **Estándar abierto** (Anthropic 2024) — cualquier modelo/cliente lo habla.
- Separa el **agente** del **proveedor de contexto**: una sola integración sirve para Claude, GPT, etc.
- Tres primitivas: **tools** (acciones), **resources** (datos), **prompts** (templates).
- Transporte: stdio, HTTP+SSE, websockets.
- JSON-RPC 2.0 como protocolo.

## 2. Mini servidor MCP

In [ ]:
DOCS = {
    'rag.md':   'RAG retrieves docs and feeds them to an LLM via context.',
    'lora.md':  'LoRA fine-tunes by adding low-rank matrices to frozen weights.',
    'mcp.md':   'MCP is the Model Context Protocol, open standard by Anthropic.',
    'agents.md':'Agents loop Thought-Action-Observation (ReAct) over tools.'
}

TOOLS = {
    'get_weather': {
        'description': 'Devuelve el clima actual de una ciudad (mock).',
        'inputSchema': {'type': 'object', 'properties': {'city': {'type': 'string'}}, 'required': ['city']}
    },
    'search_docs': {
        'description': 'Busca regex en la base de documentos local.',
        'inputSchema': {'type': 'object', 'properties': {'query': {'type': 'string'}}, 'required': ['query']}
    }
}

WEATHER = {'buenos aires': '22 °C, parcialmente nublado',
           'tokyo': '15 °C, despejado',
           'reykjavik': '2 °C, lluvia'}

In [ ]:
class MCPServer:
    def handle(self, req):
        method, params, rid = req['method'], req.get('params', {}), req.get('id')
        try:
            if method == 'initialize':
                result = {'serverInfo': {'name': 'demo-mcp', 'version': '0.1.0'},
                          'capabilities': {'tools': {}, 'resources': {}}}
            elif method == 'tools/list':
                result = {'tools': [{'name': k, **v} for k, v in TOOLS.items()]}
            elif method == 'tools/call':
                name, args = params['name'], params.get('arguments', {})
                if name == 'get_weather':
                    text = WEATHER.get(args['city'].lower(), 'sin datos')
                elif name == 'search_docs':
                    rx = re.compile(args['query'], re.I)
                    hits = [f'{n}: {c}' for n, c in DOCS.items() if rx.search(c)]
                    text = '\n'.join(hits) or 'sin matches'
                else:
                    raise ValueError(f'unknown tool {name}')
                result = {'content': [{'type': 'text', 'text': text}]}
            elif method == 'resources/read':
                uri = params['uri']   # ej 'docs://lora.md'
                name = uri.split('://')[-1]
                result = {'contents': [{'uri': uri, 'mimeType': 'text/markdown', 'text': DOCS.get(name, '')}]}
            else:
                return {'jsonrpc': '2.0', 'id': rid, 'error': {'code': -32601, 'message': 'method not found'}}
            return {'jsonrpc': '2.0', 'id': rid, 'result': result}
        except Exception as e:
            return {'jsonrpc': '2.0', 'id': rid, 'error': {'code': -32000, 'message': str(e)}}

server = MCPServer()

## 3. Cliente MCP

In [ ]:
class MCPClient:
    def __init__(self, server):
        self.server = server
    def call(self, method, params=None):
        req = {'jsonrpc': '2.0', 'id': str(uuid.uuid4())[:8], 'method': method}
        if params: req['params'] = params
        resp = self.server.handle(req)
        if 'error' in resp: raise RuntimeError(resp['error'])
        return resp['result']

client = MCPClient(server)
print(json.dumps(client.call('initialize'), indent=2))
tools = client.call('tools/list')['tools']
print('\ntools disponibles:')
for t in tools: print(f"  - {t['name']}: {t['description']}")

## 4. Loop: query → tools/list → match → call → render

In [ ]:
def route(query, tools):
    """Match heurístico de query → tool. En real lo hace un LLM."""
    q = query.lower()
    if any(w in q for w in ['clima', 'weather', 'temperatura']):
        for city in WEATHER:
            if city in q: return 'get_weather', {'city': city}
        return 'get_weather', {'city': q.split()[-1]}
    return 'search_docs', {'query': query}

for query in ['¿Cuál es el clima en Tokyo?', '¿Qué es LoRA?', 'explicame MCP', 'clima reykjavik']:
    tool, args = route(query, tools)
    out = client.call('tools/call', {'name': tool, 'arguments': args})
    text = out['content'][0]['text']
    print(f'\nQ: {query}')
    print(f'  → {tool}({args})')
    print(f'  ↳ {text}')

## 5. Resources: lectura tipada

In [ ]:
for name in ['mcp.md', 'lora.md']:
    r = client.call('resources/read', {'uri': f'docs://{name}'})
    print(f"[{name}] {r['contents'][0]['text']}")

## 6. Especificación (resumen)

**Métodos core:**
- `initialize` — handshake, intercambia capabilities.
- `tools/list`, `tools/call`
- `resources/list`, `resources/read`, `resources/subscribe`
- `prompts/list`, `prompts/get`
- `notifications/*` — server→client (resource update, log).

**Transporte real (stdio):** un proceso hijo recibe JSON-RPC línea por línea por stdin y responde por stdout.

## Ejercicio guiado

1. Agregar tool `calculator(expr)` que evalúe expresiones aritméticas seguras.
2. Implementar `tools/list` con paginación (`cursor`).
3. Agregar `notifications/resources/updated` cuando cambie DOCS.
4. Conectar el cliente vía subprocess + stdio (no in-process).

## Conclusiones

- MCP convierte tools/resources en commodity entre LLMs.
- JSON-RPC + 3 primitivas (tools/resources/prompts) cubren la mayoría de integraciones.
- Reemplaza function-calling proprietario por un estándar abierto.